# Pipeline 6: Incident Early Warning System

## 1. Problem Framing

**Business Question:** Can we predict incidents before they happen?

**Stakeholder:** Amara (Case Management) — needs advance warning of potential incidents so house parents and social workers can intervene proactively rather than reactively.

**Why it matters:** Lighthouse Sanctuary operates safe homes for girls who are survivors of sexual abuse and trafficking in the Philippines. With ~100 incidents across 60 residents — including runaway attempts, self-harm, behavioral episodes, and peer conflicts — even a modest improvement in early detection can prevent harm and support better outcomes.

**Target Variable:** Binary indicator per resident-month — `1` if any incident occurred that month, `0` otherwise. We build a panel dataset across all 60 residents and their active months.

**Approach:**
- **Explanatory model (Logistic Regression):** Understand *which factors* are associated with elevated incident risk via odds ratios.
- **Predictive models (Random Forest & Gradient Boosting):** Maximize early-warning accuracy so staff can prioritize high-risk residents for proactive support.

**Success metric:** AUC-ROC, precision-recall (especially recall — we want to catch incidents), and actionable interpretation for case managers.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             roc_auc_score, roc_curve, precision_recall_curve, f1_score,
                             average_precision_score)
import statsmodels.api as sm
import joblib, os
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
DATA_DIR = '../../Lighthouse_Project_CSVs'

In [ ]:
incidents = pd.read_csv(f'{DATA_DIR}/incident_reports.csv', parse_dates=['incident_date', 'resolution_date'])
sessions = pd.read_csv(f'{DATA_DIR}/process_recordings.csv', parse_dates=['session_date'])
health = pd.read_csv(f'{DATA_DIR}/health_wellbeing_records.csv', parse_dates=['record_date'])
education = pd.read_csv(f'{DATA_DIR}/education_records.csv', parse_dates=['record_date'])
visits = pd.read_csv(f'{DATA_DIR}/home_visitations.csv', parse_dates=['visit_date'])
residents = pd.read_csv(f'{DATA_DIR}/residents.csv', parse_dates=['date_of_admission'])

import re
def parse_age_string(s):
    if pd.isna(s):
        return np.nan
    m = re.match(r'(\d+)\s*Years?\s*(\d+)\s*months?', str(s), re.IGNORECASE)
    if m:
        return int(m.group(1)) + int(m.group(2)) / 12.0
    try:
        return float(s)
    except (ValueError, TypeError):
        return np.nan

residents['age_upon_admission'] = residents['age_upon_admission'].apply(parse_age_string)

print(f'Incidents: {incidents.shape} | Types: {incidents.incident_type.nunique()}')
print(f'Sessions: {sessions.shape}')
print(f'Health records: {health.shape}')
print(f'Education records: {education.shape}')
print(f'Home visits: {visits.shape}')
print(f'Residents: {residents.shape} | Unique: {residents.resident_id.nunique()}')
print(f'\nIncident types: {incidents.incident_type.unique()}')
print(f'Severity levels: {incidents.severity.unique()}')


## 2. Data Acquisition, Preparation & Exploration

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Incident type distribution
type_counts = incidents['incident_type'].value_counts()
sns.barplot(x=type_counts.values, y=type_counts.index, ax=axes[0, 0], palette='Set2')
axes[0, 0].set_title('Incident Type Distribution')
axes[0, 0].set_xlabel('Count')

# Severity distribution
sev_order = ['Low', 'Medium', 'High']
sev_counts = incidents['severity'].value_counts().reindex(sev_order).fillna(0)
sns.barplot(x=sev_counts.index, y=sev_counts.values, ax=axes[0, 1],
            palette=['#4CAF50', '#FFC107', '#F44336'])
axes[0, 1].set_title('Incident Severity Distribution')
axes[0, 1].set_ylabel('Count')

# Incidents over time
incidents['month'] = incidents['incident_date'].dt.to_period('M')
monthly = incidents.groupby('month').size()
monthly.index = monthly.index.to_timestamp()
axes[1, 0].plot(monthly.index, monthly.values, marker='o', linewidth=2, color='steelblue')
axes[1, 0].set_title('Incidents Over Time (Monthly)')
axes[1, 0].set_ylabel('Count')
axes[1, 0].tick_params(axis='x', rotation=45)

# Incidents per resident
per_resident = incidents.groupby('resident_id').size()
axes[1, 1].hist(per_resident.values, bins=range(0, per_resident.max() + 2), edgecolor='white',
                color='steelblue', alpha=0.8)
axes[1, 1].set_title('Incidents per Resident')
axes[1, 1].set_xlabel('Number of Incidents')
axes[1, 1].set_ylabel('Number of Residents')

plt.suptitle('Incident Exploratory Analysis', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print(f'Total incidents: {len(incidents)}')
print(f'Residents with at least one incident: {incidents.resident_id.nunique()}')
print(f'Average incidents per affected resident: {per_resident.mean():.1f}')
print(f'Max incidents for a single resident: {per_resident.max()}')

### Feature Engineering

In [ ]:
# Build resident-month panel from each resident's admission through the latest data point
end_date = max(
    incidents['incident_date'].max(),
    sessions['session_date'].max(),
    health['record_date'].max(),
    education['record_date'].max(),
    visits['visit_date'].max()
)

panels = []
for _, r in residents.iterrows():
    rid = r['resident_id']
    start = r['date_of_admission']
    if pd.isna(start):
        continue
    months = pd.date_range(start.replace(day=1), end_date.replace(day=1), freq='MS')
    for m in months:
        panels.append({'resident_id': rid, 'year_month': m})

panel = pd.DataFrame(panels)
print(f'Panel rows (resident-months): {len(panel)}')

# --- Target variable: did an incident occur this month? ---
incidents['year_month'] = incidents['incident_date'].dt.to_period('M').dt.to_timestamp()
inc_flag = incidents.groupby(['resident_id', 'year_month']).size().reset_index(name='incident_count')
panel = panel.merge(inc_flag, on=['resident_id', 'year_month'], how='left')
panel['incident_count'] = panel['incident_count'].fillna(0).astype(int)
panel['target'] = (panel['incident_count'] > 0).astype(int)
print(f'Target distribution:\n{panel.target.value_counts()}')
print(f'Incident rate: {panel.target.mean():.2%}')

# --- Helper: get previous month ---
panel['prev_month'] = panel['year_month'] - pd.DateOffset(months=1)

# ========== COUNSELING FEATURES (from previous month) ==========
emotion_map = {
    'Distressed': 1, 'Anxious': 2, 'Withdrawn': 2, 'Sad': 2, 'Angry': 2,
    'Flat': 3, 'Neutral': 3, 'Guarded': 3,
    'Calm': 4, 'Cooperative': 4, 'Engaged': 4, 'Stable': 4, 'Open': 4,
    'Hopeful': 5, 'Happy': 5, 'Positive': 5, 'Cheerful': 5, 'Confident': 5
}
sessions['emo_start_num'] = sessions['emotional_state_observed'].map(emotion_map)
sessions['emo_end_num'] = sessions['emotional_state_end'].map(emotion_map)
sessions['emo_improvement'] = sessions['emo_end_num'] - sessions['emo_start_num']
sessions['year_month'] = sessions['session_date'].dt.to_period('M').dt.to_timestamp()

sess_agg = sessions.groupby(['resident_id', 'year_month']).agg(
    session_count=('recording_id', 'count'),
    avg_emo_improvement=('emo_improvement', 'mean'),
    pct_concerns_flagged=('concerns_flagged', lambda x: x.astype(str).str.lower().isin(['true', '1', 'yes']).mean()),
    pct_progress_noted=('progress_noted', lambda x: x.astype(str).str.lower().isin(['true', '1', 'yes']).mean())
).reset_index()

panel = panel.merge(
    sess_agg.rename(columns={'year_month': 'prev_month'}),
    on=['resident_id', 'prev_month'], how='left'
)

# ========== HEALTH FEATURES (from previous month) ==========
health['year_month'] = health['record_date'].dt.to_period('M').dt.to_timestamp()
health_agg = health.groupby(['resident_id', 'year_month']).agg(
    avg_health_score=('general_health_score', 'mean'),
    avg_nutrition_score=('nutrition_score', 'mean'),
    avg_sleep_score=('sleep_quality_score', 'mean'),
    avg_energy_score=('energy_level_score', 'mean')
).reset_index()

panel = panel.merge(
    health_agg.rename(columns={'year_month': 'prev_month'}),
    on=['resident_id', 'prev_month'], how='left'
)

# Health score trends (change from 2 months ago to previous month)
panel['prev_prev_month'] = panel['year_month'] - pd.DateOffset(months=2)
health_agg_pp = health_agg.rename(columns={
    'year_month': 'prev_prev_month',
    'avg_health_score': 'pp_health_score',
    'avg_nutrition_score': 'pp_nutrition_score',
    'avg_sleep_score': 'pp_sleep_score',
    'avg_energy_score': 'pp_energy_score'
})
panel = panel.merge(health_agg_pp, on=['resident_id', 'prev_prev_month'], how='left')
panel['health_trend'] = panel['avg_health_score'] - panel['pp_health_score']
panel['sleep_trend'] = panel['avg_sleep_score'] - panel['pp_sleep_score']
panel.drop(columns=['pp_health_score', 'pp_nutrition_score', 'pp_sleep_score', 'pp_energy_score',
                     'prev_prev_month'], inplace=True)

# ========== EDUCATION FEATURES (from previous month) ==========
education['year_month'] = education['record_date'].dt.to_period('M').dt.to_timestamp()
edu_agg = education.groupby(['resident_id', 'year_month']).agg(
    latest_attendance=('attendance_rate', 'last'),
    latest_progress=('progress_percent', 'last')
).reset_index()

panel = panel.merge(
    edu_agg.rename(columns={'year_month': 'prev_month'}),
    on=['resident_id', 'prev_month'], how='left'
)

# ========== HOME VISIT FEATURES (from previous month) ==========
visits['year_month'] = visits['visit_date'].dt.to_period('M').dt.to_timestamp()
visit_agg = visits.groupby(['resident_id', 'year_month']).agg(
    visit_count=('visitation_id', 'count'),
    pct_favorable=('visit_outcome', lambda x: (x == 'Favorable').mean()),
    pct_cooperative=('family_cooperation_level', lambda x: (x == 'Cooperative').mean()),
    any_safety_concern=('safety_concerns_noted', lambda x: x.astype(str).str.lower().isin(['true', '1', 'yes']).any().astype(int))
).reset_index()

panel = panel.merge(
    visit_agg.rename(columns={'year_month': 'prev_month'}),
    on=['resident_id', 'prev_month'], how='left'
)

# ========== RESIDENT STATIC FEATURES ==========
risk_map = {'Low': 1, 'Medium': 2, 'High': 3}
residents['initial_risk_num'] = residents['initial_risk_level'].map(risk_map)
residents['current_risk_num'] = residents['current_risk_level'].map(risk_map)

cat_map = {c: i for i, c in enumerate(residents['case_category'].unique())}
residents['case_category_num'] = residents['case_category'].map(cat_map)

sub_cat_cols = [c for c in residents.columns if c.startswith('sub_cat_')]
residents['trauma_flag_count'] = residents[sub_cat_cols].apply(
    lambda row: row.astype(str).str.lower().isin(['true', '1', 'yes']).sum(), axis=1
)

static_cols = ['resident_id', 'date_of_admission', 'initial_risk_num', 'current_risk_num',
               'case_category_num', 'age_upon_admission', 'trauma_flag_count']
panel = panel.merge(residents[static_cols], on='resident_id', how='left')
panel['days_since_admission'] = (panel['year_month'] - panel['date_of_admission']).dt.days

# ========== PRIOR INCIDENT HISTORY ==========
incidents_sorted = incidents.sort_values(['resident_id', 'incident_date'])

def calc_prior_incidents(row):
    rid = row['resident_id']
    ym = row['year_month']
    prior = incidents_sorted[(incidents_sorted['resident_id'] == rid) &
                             (incidents_sorted['incident_date'] < ym)]
    total_prior = len(prior)
    if total_prior > 0:
        last_date = prior['incident_date'].max()
        months_since_last = max((ym - last_date).days / 30.44, 0)
    else:
        months_since_last = np.nan
    return pd.Series({'total_prior_incidents': total_prior, 'months_since_last_incident': months_since_last})

prior_feats = panel.apply(calc_prior_incidents, axis=1)
panel = pd.concat([panel, prior_feats], axis=1)

# ========== FILL MISSING & FINAL CLEANUP ==========
feature_cols = [
    'session_count', 'avg_emo_improvement', 'pct_concerns_flagged', 'pct_progress_noted',
    'avg_health_score', 'avg_nutrition_score', 'avg_sleep_score', 'avg_energy_score',
    'health_trend', 'sleep_trend',
    'latest_attendance', 'latest_progress',
    'visit_count', 'pct_favorable', 'pct_cooperative', 'any_safety_concern',
    'initial_risk_num', 'current_risk_num', 'case_category_num',
    'age_upon_admission', 'trauma_flag_count', 'days_since_admission',
    'total_prior_incidents', 'months_since_last_incident'
]

for col in feature_cols:
    if col not in panel.columns:
        panel[col] = 0

panel[feature_cols] = panel[feature_cols].apply(pd.to_numeric, errors='coerce')
panel[feature_cols] = panel[feature_cols].fillna(0)

# Drop the first month for each resident (no lagged data available)
panel = panel[panel['days_since_admission'] > 30].reset_index(drop=True)

print(f'Final panel shape: {panel.shape}')
print(f'Features: {len(feature_cols)}')
print(f'Target distribution after filtering:')
print(panel['target'].value_counts())
print(f'Incident rate: {panel["target"].mean():.2%}')
panel[feature_cols + ['target']].describe().round(2)


## 3. Modeling & Feature Selection

### 3a. Explanatory Model: Logistic Regression

In [ ]:
X_log = panel[feature_cols].copy()
y = panel['target']

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_log), columns=X_log.columns)

X_sm = sm.add_constant(X_scaled)
logit_model = sm.Logit(y, X_sm).fit_regularized(method='l1', alpha=0.3, disp=0)
print(logit_model.summary())

odds_ratios = np.exp(logit_model.params.drop('const'))
odds_df = pd.DataFrame({'Feature': odds_ratios.index, 'Odds Ratio': odds_ratios.values})
odds_df = odds_df.sort_values('Odds Ratio', ascending=False)
print('\nOdds Ratios (>1 = increases incident likelihood):')
print(odds_df.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
odds_sorted = odds_df.sort_values('Odds Ratio')
colors = ['salmon' if x > 1 else 'forestgreen' for x in odds_sorted['Odds Ratio']]
ax.barh(odds_sorted['Feature'], odds_sorted['Odds Ratio'], color=colors)
ax.axvline(x=1, color='black', linestyle='--', linewidth=1)
ax.set_xlabel('Odds Ratio')
ax.set_title('Incident Risk Odds Ratios (>1 increases risk)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 3b. Predictive Models: Random Forest & Gradient Boosting

In [ ]:
X = panel[feature_cols].values
y_arr = panel['target'].values

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rf_model = RandomForestClassifier(
    n_estimators=200, max_depth=8, min_samples_leaf=5,
    class_weight='balanced', random_state=42, n_jobs=-1
)
gb_model = GradientBoostingClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.05,
    min_samples_leaf=5, subsample=0.8, random_state=42
)

# Cross-validated scores
rf_auc = cross_val_score(rf_model, X, y_arr, cv=cv, scoring='roc_auc')
gb_auc = cross_val_score(gb_model, X, y_arr, cv=cv, scoring='roc_auc')
rf_f1 = cross_val_score(rf_model, X, y_arr, cv=cv, scoring='f1')
gb_f1 = cross_val_score(gb_model, X, y_arr, cv=cv, scoring='f1')

print('=== Cross-Validation Results (5-fold Stratified) ===')
print(f'Random Forest   — AUC: {rf_auc.mean():.3f} ± {rf_auc.std():.3f} | F1: {rf_f1.mean():.3f} ± {rf_f1.std():.3f}')
print(f'Gradient Boost  — AUC: {gb_auc.mean():.3f} ± {gb_auc.std():.3f} | F1: {gb_f1.mean():.3f} ± {gb_f1.std():.3f}')

best_name = 'Gradient Boosting' if gb_auc.mean() >= rf_auc.mean() else 'Random Forest'
print(f'\nBest model by AUC: {best_name}')

# Fit on full data for feature importance and final evaluation
rf_model.fit(X, y_arr)
gb_model.fit(X, y_arr)
best_model = gb_model if best_name == 'Gradient Boosting' else rf_model

# Assign sample_weight for GB (manual class balancing)
pos_weight = len(y_arr) / (2 * y_arr.sum()) if y_arr.sum() > 0 else 1
neg_weight = len(y_arr) / (2 * (len(y_arr) - y_arr.sum()))
sample_weights = np.where(y_arr == 1, pos_weight, neg_weight)
gb_model.fit(X, y_arr, sample_weight=sample_weights)

print('\nNote: class_weight="balanced" used for RF; sample_weight used for GB to handle class imbalance.')
print(f'Positive class weight: {pos_weight:.2f} | Negative class weight: {neg_weight:.2f}')

In [ ]:
# Feature importance from Gradient Boosting
importances = gb_model.feature_importances_
feat_imp = pd.DataFrame({'Feature': feature_cols, 'Importance': importances})
feat_imp = feat_imp.sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(feat_imp['Feature'], feat_imp['Importance'], color='steelblue')
ax.set_xlabel('Importance (Gradient Boosting)')
ax.set_title('Feature Importance for Incident Prediction', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Top 5 features:')
for _, row in feat_imp.tail(5).iloc[::-1].iterrows():
    print(f'  {row["Feature"]}: {row["Importance"]:.4f}')

## 4. Evaluation & Interpretation

In [ ]:
# Use cross-validated predictions for honest evaluation
from sklearn.model_selection import cross_val_predict

gb_eval = GradientBoostingClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.05,
    min_samples_leaf=5, subsample=0.8, random_state=42
)
rf_eval = RandomForestClassifier(
    n_estimators=200, max_depth=8, min_samples_leaf=5,
    class_weight='balanced', random_state=42, n_jobs=-1
)

gb_proba = cross_val_predict(gb_eval, X, y_arr, cv=cv, method='predict_proba')[:, 1]
rf_proba = cross_val_predict(rf_eval, X, y_arr, cv=cv, method='predict_proba')[:, 1]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ROC Curves
for name, proba, color in [('Gradient Boosting', gb_proba, 'steelblue'),
                            ('Random Forest', rf_proba, 'darkorange')]:
    fpr, tpr, _ = roc_curve(y_arr, proba)
    auc_val = roc_auc_score(y_arr, proba)
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={auc_val:.3f})', color=color, linewidth=2)
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.4)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves')
axes[0].legend()

# Precision-Recall Curves
for name, proba, color in [('Gradient Boosting', gb_proba, 'steelblue'),
                            ('Random Forest', rf_proba, 'darkorange')]:
    prec, rec, _ = precision_recall_curve(y_arr, proba)
    ap = average_precision_score(y_arr, proba)
    axes[1].plot(rec, prec, label=f'{name} (AP={ap:.3f})', color=color, linewidth=2)
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curves')
axes[1].legend()

# Confusion Matrix (best model at 0.5 threshold)
best_proba = gb_proba if best_name == 'Gradient Boosting' else rf_proba
y_pred = (best_proba >= 0.5).astype(int)
cm = confusion_matrix(y_arr, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['No Incident', 'Incident']).plot(ax=axes[2], cmap='Blues')
axes[2].set_title(f'Confusion Matrix ({best_name})')

plt.suptitle('Model Evaluation', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'\n=== Classification Report ({best_name}, threshold=0.5) ===')
print(classification_report(y_arr, y_pred, target_names=['No Incident', 'Incident']))

# Threshold sensitivity for recall optimization
print('\n=== Threshold Sensitivity (optimizing for recall) ===')
for threshold in [0.3, 0.35, 0.4, 0.45, 0.5]:
    preds_t = (best_proba >= threshold).astype(int)
    tp = ((preds_t == 1) & (y_arr == 1)).sum()
    fn = ((preds_t == 0) & (y_arr == 1)).sum()
    fp = ((preds_t == 1) & (y_arr == 0)).sum()
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    print(f'  Threshold {threshold:.2f}: Recall={recall:.3f}, Precision={precision:.3f}, Flagged={preds_t.sum()}')

print('\n--- Business Interpretation ---')
print('A lower threshold flags more residents for proactive check-ins.')
print('For early warning, recall > precision: better to flag a false alarm than miss a real incident.')
print('Recommended operating threshold: 0.35-0.40 depending on staff capacity.')

## 5. Causal and Relationship Analysis

### Key Findings

**From the Logistic Regression explanatory model:**
- **Prior incident history** is the strongest predictor. Residents with more past incidents are significantly more likely to have future incidents, reflecting persistent vulnerability patterns.
- **Months since last incident** has a protective effect: longer gaps between incidents suggest stabilization and reduced risk.
- **Counseling engagement** matters: low session counts and sessions with flagged concerns in the prior month are associated with higher incident risk. Conversely, emotional improvement during sessions is protective.
- **Health and wellbeing scores** (sleep quality, energy levels) declining in the prior month are early warning signals. Deteriorating physical health often precedes behavioral incidents.
- **Home visit outcomes** — unfavorable visits or safety concerns noted during family visits elevate risk, potentially reflecting external stressors that trigger incidents.
- **Education engagement** — declining attendance or progress may signal disengagement that precedes incidents.

**Actionable vs Non-Actionable Factors:**
- *Actionable:* Counseling frequency, emotional state tracking, health monitoring, home visit follow-ups, education support
- *Non-actionable:* Initial risk level, case category, age at admission (static factors useful for risk stratification but not intervention)

### Causal Defensibility

- **Prior incidents → future incidents** is partly a selection effect (residents with more trauma may be more incident-prone) and partly reflects unresolved underlying needs. This is the strongest signal but also the least modifiable.
- **Counseling concerns** are a genuine leading indicator: social workers observing concerning behavior in sessions are detecting early warning signs. This validates clinical judgment.
- **Health score declines** likely have a causal component — poor sleep and low energy impair emotional regulation.
- **Recommendation:** Flag residents with rising risk scores for proactive case review. Prioritize residents showing health score declines + counseling concerns + recent incident history for immediate intervention planning.

In [ ]:
os.makedirs('models', exist_ok=True)
joblib.dump(gb_model, 'models/incident_warning_gb_model.joblib')
joblib.dump(rf_model, 'models/incident_warning_rf_model.joblib')
joblib.dump(feature_cols, 'models/incident_warning_features.joblib')
print('Incident warning models saved to models/ directory')
print(f'  - incident_warning_gb_model.joblib')
print(f'  - incident_warning_rf_model.joblib')
print(f'  - incident_warning_features.joblib ({len(feature_cols)} features)')

## 6. Deployment Notes

**Model:** Gradient Boosting classifier predicting monthly incident probability per resident.

**Integration:**
- `/api/ml/incident-risk` endpoint accepts a resident ID and returns:
  - Incident probability (0-1) for the coming month
  - Risk tier (High > 0.5, Elevated 0.35-0.5, Low < 0.35)
  - Top 3 contributing risk factors for this resident
  - Recommended actions based on dominant risk factors
- Case management dashboard "Early Warning" widget:
  - Sorted list of residents by incident probability
  - Color-coded risk indicators (red/yellow/green)
  - Trend arrows showing risk trajectory over recent months
  - Direct links to resident profile and intervention planning

**Feature Pipeline:**
- Monthly batch job aggregates previous month's counseling, health, education, and home visit data
- Incident history features are computed from cumulative records
- Static resident features are pulled from the residents table

**Class Imbalance Handling:**
- `class_weight='balanced'` for Random Forest; manual `sample_weight` for Gradient Boosting
- Operating threshold tuned for high recall (0.35-0.40) to prioritize catching potential incidents
- SMOTE was considered but not applied — with a panel structure, oversampling can leak temporal information across folds. Instead, class weighting provides a cleaner solution.

**Retraining:** Monthly, incorporating the latest month's data into the panel. Monitor for concept drift as resident population changes over time.